# Memory Information

In [ ]:
import psutil
def get_size(bytes, suffix="B"):
    factor = 1024
    for unit in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f}{unit}{suffix}"
        bytes /= factor
print("="*40, "Memory Information", "="*40)
svmem = psutil.virtual_memory()
print(f"Total: {get_size(svmem.total)}") ; print(f"Available: {get_size(svmem.available)}")
print(f"Used: {get_size(svmem.used)}") ; print(f"Percentage: {svmem.percent}%")

# GPU Information

In [ ]:
! nvidia-smi

# Preparing Colab

## Installing packages


In [ ]:
!apt install tree neofetch
!neofetch

In [ ]:
!pip install --upgrade pip
# !pip install tensorflow 
!pip install kaggle opencv-python sklearn albumentations nltk --upgrade 
!pip install -U tensorboard_plugin_profile 
!pip install tensorflow-addons

## Mount DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# dirs
KAGGLE = "drive/My Drive/NRB_project/kaggle.json" # key to download the data
drive_data_dir = "drive/My Drive/NRB_project/"
data_dir = "./data"
checks_dir = "./checkpoints"
csv_path = "./data/data.csv"

weights_dir = "drive/My Drive/NRB_project/U_net"
loading_dir = './U-Net_Summary_conv-net/'

In [ ]:
! mkdir ~/.kaggle
! mkdir ./data
! mkdir ./checkpoints


## Download Data

In [ ]:
! cp "{KAGGLE}" ~/.kaggle/ # copy kaggle key
!kaggle datasets download -d tapakah68/yandextoloka-water-meters-dataset 


In [ ]:
! unzip -q yandextoloka-water-meters-dataset.zip -d data # unzipping
!mv data/WaterMeters/* data/                             # moving files to /data
!rm -r data/WaterMeters/                                 # removing empty folder

In [ ]:
!ls "{data_dir}/"

# Imports

In [ ]:
import os
import cv2
import pprint
import random
import itertools
import pandas as pd
import tensorflow as tf

import tensorflow_addons as tfa

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

plt.style.use('ggplot')

import albumentations as A
import math
import time
import nltk

from collections import Counter
from sklearn.model_selection import train_test_split

def asHHMMSS(s):
    m = math.floor(s / 60)
    s -= m * 60
    h = math.floor(m / 60)
    m -= h * 60
    return '%dh:%dm:%ds'% (h, m, s)



print(f"Albumentations v: {A.__version__}")
print(f"TensorFlow v: {tf.__version__}")
asHHMMSS(172)

In [ ]:
# tf.test.gpu_device_name()
# import tensorflow as tf
# print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

## Hyperparameters

In [ ]:
SEED = 1111

IMAGES_path = os.path.join(data_dir, "images")
MASK_path = os.path.join(data_dir, "masks")

# Extra functions

In [ ]:
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

def load_from_csv(path, delimiter=','):
    """
    Load csv file and return a NumPy array of its data

    Parameters
    ----------
    path: str
        The path to the csv file to load
    delimiter: str (default: ',')
        The csv field delimiter

    Return
    ------
    D: array
        The NumPy array of the data contained in the file
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File '{path}' does not exists.")
    return pd.read_csv(path, delimiter=delimiter, engine='python')
    # return pd.read_csv(path, delimiter=delimiter, float_precision='high', encoding='utf-8' )#, engine='python')
    # return pd.read_excel(path, header=None, names=['mails'])

def visualize(colormap='viridis', figsize=(16, 7), colorbar=None, **images):
    """PLot images in one row."""
    n = len(images)
    plt.figure(figsize=figsize)
    # plt.figure(figsize=(16, 10))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.axis("off")
        if isinstance(image, list):
            plt.title(' '.join(name.split('_')).title() +": " + detokenize(image[1]), fontsize=16)        
            plt.imshow(image[0], cmap=colormap)
        else:
            plt.title(' '.join(name.split('_')).title(), fontsize=16)        
            plt.imshow(image, cmap=colormap)        
        if colorbar:
            plt.colorbar()
    plt.show()

# helper function for data visualization    
def denormalize(x, BGR_RGB=True):
    """Scale image to range 0..1 for correct plot"""
    x_max = np.percentile(x, 98)
    x_min = np.percentile(x, 2)    
    x = (x - x_min) / (x_max - x_min)
    x = x.clip(0, 1) 
    if BGR_RGB:
        x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    return x
def normalize_0_255(img):
    '''Return array with values in range [0, 255]'''
    norm_image = cv2.normalize(img, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)
    return norm_image.astype(np.uint8)

def crop_with_argwhere(image, mask):
    # https://codereview.stackexchange.com/questions/132914/crop-black-border-of-image-using-numpy
    # mask should has values > 0
    # Coordinates of non-black pixels.
    coords = np.argwhere(mask)
    # Bounding box of non-black pixels.
    x0, y0 = coords.min(axis=0)
    x1, y1 = coords.max(axis=0) + 1   # slices are exclusive at the top
    # Get the contents of the bounding box.
    cropped = image[x0:x1, y0:y1]
    return cropped

In [ ]:
# classes for data loading and preprocessing
class Dataset:
    """ Water-meter Dataset. Read images, 
        apply augmentation and preprocessing transformations.
    
    Args:
        images_dir (str): path to images folder
        masks_dir (str): path to segmentation masks folder
        augmentation (albumentations.Compose): data transfromation pipeline 
            (e.g. flip, scale, etc.)
        preprocessing (albumentations.Compose): data preprocessing 
            (e.g. noralization, shape manipulation, etc.)    
    """
    
    def __init__(
            self, 
            images_dir, 
            masks_dir=None, 
            augmentation=None,
            extra_augmen=None, 
            preprocessing=None
            ):
        
        # self.ids = sorted(os.listdir(images_dir))
        self.ids = images_dir #os.listdir(images_dir)
        self.masks_ids = masks_dir
        self.images_fps = [os.path.join(IMAGES_path, image_id) for image_id in self.ids]
        self.masks_fps = [os.path.join(masks_dir, image_id) for image_id in self.ids]

        self.augmentation = augmentation
        self.extra_augmen = extra_augmen # if extra_augmen else augmentation
        self.preprocessing = preprocessing
        self._augmented = []
        # self.dim = (360, 480)
        self.dim = (360*2, 480*2)

    
    def __getitem__(self, i):
        
        # read data
        image = cv2.imread(self.images_fps[i])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.dim, interpolation=cv2.INTER_NEAREST) # resizing

        # if isinstance(self.masks_ids, str): # NOT USING

        mask = cv2.imread(self.masks_fps[i], 0) # 0 = cv2.COLOR_GRAYSCALE
        mask = cv2.resize(mask, self.dim, interpolation=cv2.INTER_NEAREST) # resizing  
        
        # mask = np.array(mask).astype(np.float32)
        # extract certain classes from mask (e.g. cars)
        # masks = [(mask == v) for v in self.class_values]
        # mask = np.stack(masks, axis=-1).astype('float')
        
        # add background if mask is not binary
        # if mask.shape[-1] != 1:
        #     print('it is not bin')
        #     print(mask.shape, mask.shape[-1], mask.max(), mask.min())
        #     background = 1 - mask.sum(axis=-1, keepdims=True)
        #     mask = np.concatenate((mask, background), axis=-1)
        masks = [(mask >= 200) ]
        mask = np.stack(masks, axis=-1).astype(np.float32) #'float')
        
        # add background if mask is not binary
        if mask.shape[-1] != 1:
            background = 1 - mask.sum(axis=-1, keepdims=True)
            mask = np.concatenate((mask, background), axis=-1)
        # apply augmentations
        if self.augmentation:
            if self.images_fps[i] not in self._augmented: # check if augmented    
                sample = self.augmentation(image=image, mask=mask)
                self._augmented.append(self.images_fps[i])  
            else:              
                sample = self.extra_augmen(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']
        # apply preprocessing
        if self.preprocessing:
            sample = self.preprocessing(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask
        
    def __len__(self):
        return len(self.ids)

class Dataset_valid:        
    def __init__(
            self, 
            images_dir, 
            masks_dir=None, 
            augmentation=None,
            preprocessing=None
            ):
        self.ids = images_dir 
        self.masks_ids = masks_dir
        self.images_fps = [os.path.join(IMAGES_path, image_id) for image_id in self.ids]
        self.masks_fps = [os.path.join(masks_dir, image_id) for image_id in self.ids]

        self.augmentation = augmentation
        self.preprocessing = preprocessing
        self._augmented = []
        self.dim = (360*2, 480*2)

    
    def __getitem__(self, i):        
        # read data
        image = cv2.imread(self.images_fps[i])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.dim, interpolation=cv2.INTER_NEAREST) # resizing

        mask = cv2.imread(self.masks_fps[i], 0) # 0 = cv2.COLOR_GRAYSCALE
        mask = cv2.resize(mask, self.dim, interpolation=cv2.INTER_NEAREST) # resizing  
        
        masks = [(mask >= 200) ]
        mask = np.stack(masks, axis=-1).astype(np.float32) #'float')
        
        # add background if mask is not binary
        if mask.shape[-1] != 1:
            background = 1 - mask.sum(axis=-1, keepdims=True)
            mask = np.concatenate((mask, background), axis=-1)
        # apply augmentations
        if self.augmentation:            
            sample = self.augmentation(image=image, mask=mask)               
            image, mask = sample['image'], sample['mask']
        # apply preprocessing
        if self.preprocessing:
            sample = self.preprocessing(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask
        
    def __len__(self):
        return len(self.ids)

class Dataloader(tf.keras.utils.Sequence):
    """Load data from dataset and form batches
    
    Args:
        dataset: instance of Dataset class for image loading and preprocessing.
        batch_size: Integet number of images in batch.
        shuffle: Boolean, if `True` shuffle image indexes each epoch.
    """
    
    def __init__(self, dataset, batch_size=1, shuffle=False):
        
        self.batch_size = batch_size
        self.shuffle = shuffle
        # if isinstance(dataset, list):
        #     self.dataset = list(itertools.chain.from_iterable(dataset))
        #     # self.dataset = itertools.chain.from_iterable(dataset)
        #     # self.indexes = np.arange(np.sum([len(d) for d in dataset] ) )
        #     self.indexes = np.arange(len(self.dataset) )
        # else:
        self.dataset = dataset
        self.indexes = np.arange(len(dataset))

        self.on_epoch_end()

    def __getitem__(self, i):
        
        # collect batch data
        start = i * self.batch_size
        stop = (i + 1) * self.batch_size
        data = []
        for j in range(start, stop):
            data.append(self.dataset[j])
        
        # transpose list of lists
        batch = [np.stack(samples, axis=0) for samples in zip(*data)]
        return tuple(batch)    
    def __len__(self):
        """Denotes the number of batches per epoch"""
        return len(self.indexes) // self.batch_size   

    def on_epoch_end(self):
        """Callback function to shuffle indexes each epoch"""
        if self.shuffle:
            self.indexes = np.random.permutation(self.indexes)
        self.dataset._augmented = [] # reset list

In [ ]:
# isinstance([dataset, testDataset], list)
# sets = list(itertools.chain.from_iterable([dataset, testDataset]))
# np.arange(np.sum([len(i) for i in [dataset, testDataset]] ) )

## Augmentation & Preprocessing

In [ ]:
def round_clip_0_1(x, **kwargs):
    return x.round().clip(0, 1)

def get_training_augmen_extra():
    train_transform = [
                A.Resize( 720, 720, interpolation=cv2.INTER_NEAREST), # properly working
                # A.Resize( 640, 640, interpolation=cv2.INTER_NEAREST),
                A.Rotate(p=.45, border_mode=0, limit=70, interpolation=cv2.INTER_NEAREST),
                A.Lambda(mask=round_clip_0_1),            
        ]
    return A.Compose(train_transform)

def get_training_augmentation():
    train_transform = [        
        A.ShiftScaleRotate(p=.5, border_mode=0, 
                            rotate_limit=15,),

        # A.PadIfNeeded(min_height=320*2, min_width=320*2, # last used
        A.PadIfNeeded(min_height=720, min_width=720,  # properly working
        # A.PadIfNeeded(min_height=320, min_width=320, 
                      always_apply=True, border_mode=0),
        # A.CenterCrop (height=320, width=320, always_apply=True),
        A.RandomCrop(height=720, width=720, always_apply=True), # properly working
        # A.CropNonEmptyMaskIfExists (height=320, width=320, always_apply=True),
        # A.CropNonEmptyMaskIfExists (height=640, width=320*2, always_apply=True), # last used
        # A.CropNonEmptyMaskIfExists (height=720, width=720, always_apply=True),
                 
        A.IAAAdditiveGaussianNoise(p=0.2),
        # A.IAAPerspective(p=0.5), # not good for now
        A.OneOf( # problem `HueSaturationValue` does not work for gray images
            [
                A.RandomContrast(p=1),
                A.HueSaturationValue(p=1),
            ],
            p=0.77,
        ),
        A.OneOf(
            [
                A.CLAHE(p=1),
                A.RandomBrightness(p=1),
                A.RandomGamma(p=1),
            ],
            p=0.77,
        ),
        A.OneOf(
            [
                A.IAASharpen(p=1),
                A.Blur(blur_limit=3, p=1),
                A.MotionBlur(blur_limit=3, p=1),
            ],
            p=0.77,
        ),
        # A.Rotate(limit=50, p=0.33, border_mode=0),
        A.Lambda(mask=round_clip_0_1),
        
    ]
    return A.Compose(train_transform)

def preprocess_data_input(image=None, mask=None, **kwargs):
    """Preprocess Images according to `ImageNet`
    Args:
        image: image to process
        mask: mask of the image
        kwargs: inserted to not break the convention of 
                albumentations.Compose
    Return:
        function from keras that process the image.    
    """
    return tf.keras.applications.imagenet_utils.preprocess_input(image.copy())

def get_validation_augmentation():
    """Add paddings to make image shape divisible by 32"""
    test_transform = [
        # A.PadIfNeeded(384, 480)
        A.PadIfNeeded(384*2, 480*2)
        # A.PadIfNeeded(640, 640)
        # A.PadIfNeeded(min_height=320, min_width=320, 
        #               always_apply=True, border_mode=0),
    ]
    return A.Compose(test_transform)

def get_preprocessing(preprocessing_fn):
    """Construct preprocessing transform
    
    Args:
        preprocessing_fn (callbale): data normalization function 
            (can be specific for each pretrained neural network)
    Return:
        transform: albumentations.Compose
    
    """    
    _transform = [
        A.Lambda(image=preprocessing_fn),
    ]
    return A.Compose(_transform)

# DATA

In [ ]:
def grab_value_from_name(sentence):
    '''Function to grab the actual value from image name; due to problems in the CSV file.'''
    start, end = "value_", ".jpg"
    return sentence[sentence.find(start)+len(start) : sentence.rfind(end)].replace("_", ".")

## running data

CSV = load_from_csv(csv_path)
print(CSV.head(10))

X = np.array(CSV['photo_name'].tolist())
y = [grab_value_from_name(name) for name in X]

print("Data shape:", X.shape, np.array(y).shape)

## Data for Segmentation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, np.array(y), test_size=0.25, random_state=SEED)
print("New Train-Test shape:", X_train.shape, X_test.shape, y_train.shape, y_test.shape)


- In the following cells, extra samples from the same training set were added to the training set.

In [ ]:
rand_samples = np.random.RandomState(SEED).choice(X_train, size=int(len(X_train)*.33), replace=0)
# rand_targets = np.random.RandomState(SEED).choice(y_train, size=int(len(X_train)*.33), replace=False)

len(rand_samples),  len(np.unique(rand_samples)) #len(rand_targets), len(np.unique(rand_targets))

- Comparing results between original and augmented results

In [ ]:
for i in range(264, 269):
    original = os.path.join( './data/images' , X_test[i])
    original = cv2.imread(original)
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
    # original = cv2.resize(original, (960, 960)) # to remove
    image, mask = dataset[i] # get some sample
    visualize(
        original=original,
        # image = normalize_to_RGBvals(image), 
        image = image, 
        mask = mask[..., 0]
        , figsize=(20,12)
        
    )
    print(original.shape, image.shape, mask.shape)

In [ ]:
dataset = Dataset(X_test, MASK_path
                  , augmentation=get_training_augmen_extra()
                #   , extra_augmen=get_training_augmen_extra()
                #   , augmentation=get_training_augmen_extra()
                #   , augmentation=get_validation_augmentation()
                #   , preprocessing=get_preprocessing(preprocess_data_input),
                  )
print("Size of Augmented dataset: {}".format(len(dataset)))

for i in range(264, 269):
    image, mask = dataset[i] # get some sample
    visualize(
        # image = normalize_to_RGBvals(image), 
        image = image, 
        mask = mask[..., 0]
    )
    plt.figure(figsize=(16, 7))
    plt.imshow(image)
    plt.imshow(mask.squeeze(), alpha=.6, cmap='copper', interpolation=None) # copper
    plt.axis('off');
    print(image.shape, mask.shape)


- Observing random samples from Train set

In [ ]:
X_train.shape, rand_samples.shape, np.concatenate((X_train, rand_samples)).shape

In [ ]:
dataset = Dataset(np.concatenate((X_train, rand_samples)), MASK_path
                  , augmentation=get_training_augmentation()
                  , extra_augmen=get_training_augmen_extra()
                #   , preprocessing=get_preprocessing(preprocess_data_input),
                  )
print("Size of Augmented dataset: {}".format(len(dataset)))

for i in range(len(dataset)):
    image, mask = dataset[i] # get some sample
    if i > 929 and i < 937:
        visualize(
            image = image, 
            mask = mask[..., 0]
        )
        plt.figure(figsize=(16, 7))
        plt.imshow(image)
        plt.imshow(mask.squeeze(), alpha=.6, cmap='copper', interpolation=None) # copper
        plt.axis('off');
        print(image.shape, mask.shape)

f"Repeated items: {len(dataset) - len(dataset._augmented)}"

In [ ]:
def plot_image_histograms(image): 
    # find frequency of pixels in range 0-255 
    # histr = cv2.calcHist([image],[0],None,[256],[0,256]) 
    # show the plotting graph of an image 
    fig = plt.figure(figsize=(8, 8))
    plt.subplot(2,1,1)
    plt.axis(False)
    plt.imshow(image)
    plt.subplot(2,1,2)
    
    color = ('r','g','b')
    for i, col in enumerate(color):
        histr = cv2.calcHist([image],[i],None,[256],[0,256])
        plt.plot(histr, color=col)
        plt.xlim([0,256])
    plt.show() 

plot_image_histograms(image)

# ::MODEL:: 
-  Segmentation model

In [ ]:
import itertools

tf.keras.backend.clear_session()

# IMG_WIDTH = 320  # image.shape[0]
# IMG_HEIGHT = 320 # image.shape[1]
# IMG_CHANNELS = 3 # image.shape[2]
IMG_WIDTH = image.shape[0]
IMG_HEIGHT = image.shape[1]
IMG_CHANNELS = image.shape[2]

class Unet_Model(tf.keras.Model):

    def __init__(self, activation):
        super(Unet_Model, self).__init__()

        self.activation=activation       
        self.upsTrans = [
            self._upTrans(128),
            self._upTrans(64),
            self._upTrans(32),
            self._upTrans(16)
        ]
        self.downs = [  
            self._down(16,  3, add_max_pool=False, by_strides=True),
            self._down(32,  3, add_max_pool=False, by_strides=True),
            self._down(64,  3, apply_batchnorm=True, 
                       add_max_pool=False, by_strides=True),
            self._down(128, 3, apply_batchnorm=True, 
                       add_max_pool=False, by_strides=True),
            self._down(256, 3, apply_batchnorm=True, add_max_pool=False)
        ] 
        self.downs = list(itertools.chain.from_iterable(self.downs))

        self.ups = [
            self._up(128, add_dropOut=True),
            self._up(64, True),
            self._up(32, True, .1),
            self._up(16, True, .1)
        ]
        # last layer - Sigmoid if 1 class else Softmax
        self.last = tf.keras.layers.Conv2D(filters=1, kernel_size=1
                                           , activation=self.activation
                                           , name=self.activation.title())

    def _upTrans(self, filters):
        return tf.keras.layers.Conv2DTranspose(filters, kernel_size=2,
                                               kernel_initializer='he_normal',
                                               strides=(2, 2), padding='same')
    def call(self, inputs):        
         
        x = inputs  
        skips = [] 
        for idx, layer in enumerate(self.downs, 0):            
            x = layer(x)
            if layer.name.startswith('Swish1'):
                skips.append(x)

        skips = reversed(skips[:-1])

        for l_ups, skip, l_upstrans in zip(self.ups, skips,
                                     self.upsTrans):
            x = l_upstrans(x)
            x = tf.keras.layers.Concatenate()([x, skip])
            for layer in l_ups:
                x = layer(x)
        x = self.last(x)
        return x
    
    def _down(self, filters, size, by_strides=False,
              apply_batchnorm=False, add_max_pool=True):
        
        dflt_strides = (1, 1)
        down = []        
        for i in range(2):         
            down.append(
                tf.keras.layers.Conv2D(filters=filters, kernel_size=size
                            , strides=dflt_strides , padding='same'
                            , kernel_initializer='he_normal', use_bias=False if apply_batchnorm else True
                    )  )
            if apply_batchnorm:
                down.append(tf.keras.layers.BatchNormalization())
            down.append(
                tf.keras.layers.Activation('swish'
                , name='Swish'+str(i)+'-'+str(filters)) )
            
            if i > 0 and add_max_pool is False and by_strides:
                dflt_strides = (2, 2)
                down.append(
                    tf.keras.layers.Conv2D(filters=filters, kernel_size=size
                                        , strides=dflt_strides, padding='same'
                                        , activation='swish'
                                        , kernel_initializer='he_normal'))
        if add_max_pool:
            down.append(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
        
        return down
    def _up(self, filters, add_dropOut=False, do_rate=.2):

        up = []
        up.append(tf.keras.layers.Conv2D(filters, kernel_size=3
                , kernel_initializer='he_normal', padding='same'))
        up.append(tf.keras.layers.Activation('swish') )
        if add_dropOut:
            up.append(tf.keras.layers.Dropout(do_rate))
        up.append(tf.keras.layers.Conv2D(filters, kernel_size=3
                , kernel_initializer='he_normal', padding='same'))
        up.append(tf.keras.layers.Activation('swish') )
        return up
    def summary(self):
        inputs = tf.keras.layers.Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
        return tf.keras.Model(inputs=inputs, outputs=self.call(inputs)
        , name='U-Net_Summary')#.summary()

ACTIVATION = 'sigmoid'
model = Unet_Model(ACTIVATION)

# **Training process**

## Loss 

In [ ]:
def dice_loss(y_true, y_pred):
    axis=(1,2,3) 

    numerator = 2 * tf.reduce_sum(y_true * y_pred, axis=axis)
    denominator = tf.reduce_sum(y_true + y_pred, axis=axis)
    return 1 - numerator / denominator

def focal_loss(y_true, y_pred):
    alpha=0.25
    gamma=2

    def focal_loss_with_logits(logits, targets, alpha, gamma, y_pred):
        weight_a = alpha * (1 - y_pred) ** gamma * targets
        weight_b = (1 - alpha) * y_pred ** gamma * (1 - targets)

        return (tf.math.log1p(tf.exp(-tf.abs(logits))) + tf.nn.relu(-logits)) * (weight_a + weight_b) + logits * weight_b 

    y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
    logits = tf.math.log(y_pred / (1 - y_pred))

    loss = focal_loss_with_logits(logits=logits, targets=y_true, alpha=alpha, gamma=gamma, y_pred=y_pred)
    # or reduce_sum and/or axis=-1
    return tf.reduce_mean(loss)

def loss_funct(gt, pr):   
    left = dice_loss(gt, pr)
    right = focal_loss(gt, pr)
    return tf.reduce_mean( left + (1 * right) )

## Metrics

In [ ]:
def iou_coef(y_true, y_pred, threshold=.5, smooth=1e-5, per_image=False):
    if per_image:
        axis=(1,2,3)
    else:
        axis=None
    if threshold:
        y_pred = tf.cast( tf.math.greater(y_pred, threshold), tf.float32 )

    intersection = tf.math.reduce_sum(tf.math.abs(y_true * y_pred), axis=axis ) 
    union = tf.math.reduce_sum(
        y_true, axis) + tf.math.reduce_sum(y_pred, axis ) - intersection

    iou = tf.math.reduce_mean((intersection + smooth) / (union + smooth) )# , axis=0)
    return iou

metrics= [iou_coef, 
          tfa.metrics.FBetaScore(num_classes=2, average="micro", threshold=.5)]

In [ ]:
BATCH_SIZE = 16
EPOCHS = 100
LR = 2e-4
optimizer = tf.keras.optimizers.Adam(LR)
                                     
# compile keras model with defined optimozer, loss and metrics
model.compile(
    optimizer=optimizer, loss=loss_funct, metrics=metrics
    )

## Data preparation

- Creating training and validation set
- Creating the data loaders

In [ ]:
# training images
# train_dataset = Dataset(
#     X_train, MASK_path
#     , augmentation=get_training_augmentation(),
#     preprocessing=get_preprocessing(preprocess_data_input),
# )
# training + extra original images
train_dataset = Dataset(
    np.concatenate((X_train, rand_samples)), MASK_path
    , augmentation=get_training_augmentation()
    , extra_augmen=get_training_augmen_extra()
    , preprocessing=get_preprocessing(preprocess_data_input),
)
#  validation images
valid_dataset = Dataset_valid( 
    X_test, MASK_path
    , augmentation=get_validation_augmentation(),
    preprocessing=get_preprocessing(preprocess_data_input),
)
# # data loaders
train_dataloader = Dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_dataloader = Dataloader(valid_dataset, batch_size=1, shuffle=False)

In [ ]:
len(train_dataset), len(valid_dataset) # (933, 311)

In [ ]:
im_batch, mask_batch = next(iter(train_dataloader))
im_batch.shape, mask_batch.shape, im_batch.dtype, mask_batch.dtype

In [ ]:
im_batch, mask_batch = next(iter(valid_dataloader))
im_batch.shape, mask_batch.shape, im_batch.dtype, mask_batch.dtype

## Callbacks

In [ ]:
# cleaning-up last session IF NEEDED
!rm -r U-Net_Summary_conv-net/ 
!rm -r logs_U-net

!ls

In [ ]:
dir_name = "./{}_{}".format(model.summary().name, 'conv-net')
print(f"Model checkpoints dir: {dir_name}")
!mkdir $dir_name

# define callbacks for learning rate scheduling and best checkpoints saving
callbacks = [ # all calbacks by defaul come with monitor='val_loss'
    tf.keras.callbacks.ModelCheckpoint(
                '{}/{}_{}_best_model.h5'.format(dir_name, model.summary().name,
                                        'conv-net')
                    , save_weights_only=True, save_best_only=True, mode='min'
                    , monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=10, factor=.1, mode='min'
                                        , monitor='val_loss'),
    tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True
                                , mode='min', verbose=1, monitor='val_loss'),
    tf.keras.callbacks.TensorBoard(log_dir='logs_U-net', profile_batch = '40, 55'
                                    )
            ]

## Training steps

In [ ]:
start_train = time.time() 
# bigger images run
history = model.fit(
    train_dataloader, 
    steps_per_epoch=len(train_dataloader), 
    epochs=EPOCHS, 
    callbacks=callbacks,  
    validation_data=valid_dataloader, 
    validation_steps=len(valid_dataloader)
)

print('[TRAINING FINISHED]\n'
                'Model trained during {} \n'.format( asHHMMSS( time.time() - start_train)))

In [ ]:
model.evaluate(valid_dataloader)

In [ ]:
model.load_weights('{}/{}_{}_best_model.h5'.format(dir_name, model.summary().name,
                                        'conv-net'))
model.evaluate(valid_dataloader)

In [ ]:
# !tree logs_U-net/
!tree U-Net_Summary_conv-net/

### Training visualization

In [ ]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard
%tensorboard --logdir logs_U-net

In [ ]:
def plot_loss(hist):
    plt.figure(figsize=(14, 7))

    plt.plot(hist['loss'], label='Train')
    plt.plot(hist['val_loss'], label='Valid')

    plt.annotate(f"Epoch: { np.argmin(hist['val_loss']) }; \nMin valid-loss: {min(hist['val_loss']) :.4f}"
                , xy =( np.argmin(hist['val_loss'] ) , min( hist['val_loss'] ) ), 
                textcoords="offset points", # how to position the text
                 xytext=( -20 , 50), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                arrowprops = dict(facecolor ='green', 
                                  shrink = 0.05), fontsize=12
            )
    plt.xlabel('Epochs', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.legend(fontsize=14)

plot_loss(history.history)

In [ ]:
def plot_evaluations(hist):
    f, (ax1, ax2) = plt.subplots(1, 2, figsize=(26, 7))

    ax1.plot(hist['iou_coef'],  '--', label='Train')
    ax1.plot(hist['val_iou_coef'],  '-', label='Valid')
    ax1.plot( np.argmax(hist['val_iou_coef']), max(hist['val_iou_coef']), linestyle="None", color="red", marker="X"
        , markersize=10, label='Max IOU coeff')
    
    ax1.annotate(f"Epoch: {np.argmax(hist['val_iou_coef'])}; \nIOU coeff: {max(hist['val_iou_coef']) :.4f}"
                , xy =(np.argmax(hist['val_iou_coef']) , max(hist['val_iou_coef'])), 
                textcoords="offset points", # how to position the text
                #  xytext=( 5 , 50), # distance from text to points (x,y)
                 xytext=( -22 , -50), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                # arrowprops = dict(facecolor ='green', 
                #                   shrink = 0.05)
                # va='bottom',
                fontsize=12
             ) 
    ax1.set_title(label='IOU coefficient')
    ax1.set_xlabel('Epochs', fontsize=14)
    ax1.legend(fontsize=14)

    ax2.plot(hist['fbeta_score'],  '--',  label='Train')
    ax2.plot(hist['val_fbeta_score'],  '-', label='Valid')
    ax2.plot( np.argmax(hist['val_fbeta_score']), max(hist['val_fbeta_score']), linestyle="None", color="red", marker="X"
        , markersize=10, label=r'Max $\mathcal{F}_\beta$ score')
    ax2.annotate(f"Epoch: {np.argmax(hist['val_fbeta_score'])};\n" + r"$\mathcal{F}_\beta$" + f" score: {max(hist['val_fbeta_score']) :.4f}"
                , xy =(np.argmax(hist['val_fbeta_score']) , max(hist['val_fbeta_score'])), 
                textcoords="offset points", # how to position the text
                #  xytext=( 5 , 50), # distance from text to points (x,y)
                 xytext=( -22 , -50), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                fontsize=12
             ) 

    ax2.set_title(label=r'$\mathcal{F}_\beta$ score')
    ax2.set_xlabel('Epochs', fontsize=14)
    ax2.legend(fontsize=14)

plot_evaluations(history.history)

## Weights

In [ ]:
!ls "{weights_dir}"
!ls "{loading_dir}"

### Saving Weights

Copying files to DRIVE

In [ ]:
!cp -vr U-Net_Summary_conv-net "{weights_dir}"
!cp -vr logs_U-net "{weights_dir}"

### Loading Weights

Copying files from DRIVE to COLAB

In [ ]:
!cp -vr "{weights_dir}/." ./
!ls

In [ ]:
!ls ./checkpoints/

In [ ]:
model.built = True
model.load_weights( '{}/{}_{}_best_model.h5'.format(loading_dir, 'U-Net_Summary', 'conv-net') )
model.evaluate(valid_dataloader)

In [ ]:
model.evaluate(train_dataloader)

In [ ]:
# 311/311 [==============================] - 12s 39ms/step - loss: 0.1040 - iou_coef: 0.8561 - fbeta_score: 0.9135 # old pass
# 311/311 [==============================] - 11s 35ms/step - loss: 0.1000 - iou_coef: 0.8573 - fbeta_score: 0.9167
model.evaluate(valid_dataloader) # 14s 43ms/step - loss: 0.0986 - iou_coef: 0.8570 - fbeta_score: 0.9099   

## Looking at some examples

In [ ]:

test1 = ["00883201372079_0.jpg", "02800000312555_0.jpg", "00882401101858_0.jpg", "00882401146713_0.jpg", "00883201360504_0.jpg" ]

for img in test1:
    image1 = cv2.imread(img)

    image1 = cv2.resize(image1, (720, 720))
    image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)

    image1 = tf.keras.applications.imagenet_utils.preprocess_input(image1)
    image1 = np.expand_dims(image1, axis=0)

    # pr_mask = model.predict(image1 )

    plt.figure(figsize=(16, 7))
    plt.imshow(denormalize(image1.squeeze(), True))
    pr_mask = model.predict(image1).squeeze()#.round()
    # pr_mask = np.ma.masked_where(pr_mask > .5, pr_mask)

    plt.imshow(pr_mask, alpha=.6, cmap='copper', interpolation=None) # copper
    plt.axis('off');
    plt.show()
    # visualize(
    #     original = denormalize( image1.squeeze(), BGR_RGB=False),
    #     mask = pr_mask.squeeze()
    # )

In [ ]:
data_to_plot = valid_dataset
n = 25
ids = np.random.choice(np.arange(len(data_to_plot)), size=n)

for i in ids:
    
    image, gt_mask = data_to_plot[i]
    
    image = np.expand_dims(image, axis=0)
    pr_mask = model.predict(image) #.round()
    # print(image.shape, pr_mask.shape, gt_mask.squeeze().shape)
    visualize(
        image=denormalize(image.squeeze()),
        ground_Truth_mask=gt_mask.squeeze(),
        predicted_mask=pr_mask.squeeze(),
    )
    print(f"Shapes: {image.shape} gt: {gt_mask.shape} pred: {pr_mask.shape}")

In [ ]:
plt.figure(figsize=(10,10))

# image, _ = data_to_plot[ids[n-4]]
image, _ = data_to_plot[ids[n-8]]
# image, _ = valid_dataset[34]#ids[n-2]]
print('Image shape:', image.shape)
image = np.expand_dims(image, axis=0)
plt.imshow(denormalize(image.squeeze(), True))
pr_mask = model.predict(image).squeeze().round()
# pr_mask = np.ma.masked_where(pr_mask > .5, pr_mask)

plt.imshow(pr_mask, alpha=.6, cmap='copper', interpolation='nearest') # copper
plt.axis('off');

In [ ]:
im2save = cv2.cvtColor(image.squeeze(), cv2.COLOR_BGR2RGB)
mk2save = model.predict(image).squeeze(0)
n = 5
tf.keras.preprocessing.image.save_img(f"image{n}.png", im2save)
tf.keras.preprocessing.image.save_img(f"mask{n}.png", mk2save)

In [ ]:
np.max(mk2save), np.min(mk2save)

# Port to Tensorflow Lite

In [ ]:
tf.saved_model.save(model.summary(), checks_dir)

!ls "{checks_dir}"

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(checks_dir)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite ops.
        tf.lite.OpsSet.SELECT_TF_OPS # enable TensorFlow ops.
    ]

In [ ]:
converter.target_spec.supported_types = [tf.float16]
tf_lite_model = converter.convert()
# saving the model
open(f'roi_detector_tf_float16.tflite', 'wb').write(tf_lite_model)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
converter.target_spec.supported_types = [tf.float32]
tf_lite_model = converter.convert()
# saving the model
open(f'roi_detector_tf_float32.tflite', 'wb').write(tf_lite_model)

In [ ]:
!du -sh roi_detector_tf_float16.tflite
!du -sh roi_detector_tf_float32.tflite

In [ ]:
original = os.path.join( './data/images' , X_test[ids[n-10]])
# original = os.path.join( './data/images' , X_test[238])
original = cv2.imread(original)
dim = (360*2, 480*2)
print(dim, original.shape)
original = cv2.resize(original, dim, interpolation= cv2.INTER_NEAREST)

ori_mask = os.path.join( './data/masks' , X_test[238])
ori_mask = cv2.imread(ori_mask)
ori_mask = cv2.resize(ori_mask, dim, interpolation=cv2.INTER_NEAREST)

original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
# original.shape
# original = cv2.resize(original, (720, 720)) # to remove
# original = tf.keras.applications.imagenet_utils.preprocess_input(original.copy()) 

# trans = A.CropNonEmptyMaskIfExists (height=720, width=720, always_apply=True,)(image=original, mask=ori_mask)['image']
# trans = A.ShiftScaleRotate(p=1, border_mode=0, rotate_limit=15,)(image=original)['image']
# trans = A.RandomCrop (height=720, width=720, always_apply=True,)(image=trans)['image']
trans = A.Resize( 720, 720, interpolation=cv2.INTER_NEAREST)(image=original)['image']
trans = A.Rotate(p=1.0,border_mode=0,limit=70, interpolation=cv2.INTER_NEAREST)(image=trans)['image']

# visualize(img=trans, original =original)
trans.shape
original = tf.keras.applications.imagenet_utils.preprocess_input(trans.copy()) 
original = np.expand_dims(original, axis=0)
# predict region-of-interes
output = infere_tf_lite_model("roi_detector_tf_float32.tflite", image=original)

visualize(
    image=  denormalize(original.squeeze()),
    # ground_truth= gt_mask,
    predicted_mask = output.squeeze()
)
# original.shape, output.shape

In [ ]:
# trans = A.PadIfNeeded(min_height=720, min_width=720, 
#               always_apply=True, border_mode=0)(image=original)['image']

trans = A.CropNonEmptyMaskIfExists (height=720, width=720, always_apply=True,)(image=original, mask=ori_mask)['image']

visualize(
    original = original,
    image=trans,
    mask=ori_mask
)
original.shape, trans.shape, ori_mask.shape

In [ ]:
def infere_tf_lite_model(path, image):
    input_data = image

    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()

    # Get input and output tensors.
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_shape = input_details[0]['shape']
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])
    return output

# grab image from valid dataset
image, gt_mask = valid_dataset[37]
# image = cv2.resize(image, (320, 320))
# image = cv2.resize(image, (320*2, 320*2))
image = cv2.resize(image, (720, 720))
# gt_mask = cv2.resize(gt_mask, (320, 320))
gt_mask = cv2.resize(gt_mask, (320*2, 320*2))
image = np.expand_dims(image, axis=0)
# predict region-of-interes
output = infere_tf_lite_model("roi_detector_tf_float32.tflite", image=image)
# plots
visualize(
    image= denormalize ( image.squeeze()),
    ground_truth= gt_mask,
    predicted_mask = output.squeeze()
)
image.shape, output.shape

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(denormalize(image.squeeze()))
# plt.imshow(image.squeeze(), alpha=.2)
plt.imshow(output.squeeze(), alpha=.45, cmap='copper')
plt.axis('off');

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(denormalize(image.squeeze()))
# plt.imshow(image.squeeze(), alpha=.2)
plt.imshow(output.squeeze(), alpha=.45, cmap='copper')
plt.axis('off');

In [ ]:
# copy LITE model to DRIVE
!cp -vr roi_detector_tf_float16.tflite "{weights_dir}"
!cp -vr roi_detector_tf_float32.tflite "{weights_dir}"

# Model summary & graph model

In [ ]:
model.summary().summary()

In [ ]:
tf.keras.utils.plot_model(model.summary(), show_shapes=True)